# MiniJev Kaggle job runner
Clones the **`jev`** branch, installs dependencies, finds the trained LoRA adapter, and runs
`python -m scripts.kaggle_job`. **Which job runs is decided in git** (`scripts/kaggle_job.py`),
so new jobs never need notebook edits: just **Save Version → Save & Run All**.

**Settings:** Accelerator = **GPU T4 x2**, Internet = **On**.
**Input:** the *Banking77 LoRA Finetuning* notebook version holding `banking77-lora-checkpoints/`.
Results land under `minijev-output/` in this version's Output tab.

In [ ]:
import os
import shutil
import socket
import subprocess

try:
    socket.gethostbyname('github.com')
except OSError as e:
    raise RuntimeError('No internet: turn Internet On in notebook settings.') from e

os.chdir('/kaggle/working')
shutil.rmtree('repo', ignore_errors=True)
subprocess.run(
    ['git', 'clone', '--branch', 'jev',
     'https://github.com/V1cky16112003/banking77-lora-finetuning.git', 'repo'],
    check=True,
)
os.chdir('repo')
subprocess.run(['git', 'log', '-1', '--oneline'], check=True)

In [ ]:
import sys

reqs = [line.split('#')[0].strip() for line in open('requirements.txt')]
reqs = [r for r in reqs if r and not r.startswith('torch')]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *reqs], check=True)

In [ ]:
from pathlib import Path

os.environ['CUDA_VISIBLE_DEVICES'] = '0'

# compare_readout loads an fp16 base, so prefer the non-QLoRA adapters, best-first guess.
PREFERRED = ['r16_a32_all', 'r16_a16_qv', 'r8_a16_qv']
found = {p.parent.name: p.parent for p in Path('/kaggle/input').glob('**/banking77-lora-checkpoints/*/adapter_config.json')}
print('Adapters found:', sorted(found))
ADAPTER_DIR = next((found[name] for name in PREFERRED if name in found), None)
if ADAPTER_DIR is None:
    raise RuntimeError(
        'No finished non-QLoRA adapter under /kaggle/input. Attach the training notebook '
        'version whose output contains banking77-lora-checkpoints/.'
    )
print('Using', ADAPTER_DIR)

In [ ]:
OUT = Path('/kaggle/working/minijev-output')
subprocess.run(
    [sys.executable, '-m', 'scripts.kaggle_job', '--adapter-dir', str(ADAPTER_DIR), '--output-dir', str(OUT)],
    check=True,
)